# Case Search Playground (DB function 기반)

- 검색은 DB 함수 `search_hybrid_rrf(...)`만 호출
- 상담/해결/조정은 **서브에이전트(case_subagent)** 가 반복 호출 + 병합
- 출력은 **chunk text 중심** (슈퍼바이저가 요약)


In [15]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

from dotenv import load_dotenv


# backend 찾기 (상위로 올라가며 탐색)
def find_backend_dir(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(10):
        cand = cur / "backend"
        if (cand / ".env").exists():
            return cand
        cand2 = cur / "LLM" / "backend"
        if (cand2 / ".env").exists():
            return cand2
        cur = cur.parent
    raise FileNotFoundError("backend/.env를 찾지 못했습니다.")


BACKEND = find_backend_dir(Path.cwd())
load_dotenv(BACKEND / ".env")

retrieval_dir = BACKEND / "app" / "agents" / "retrieval"
module_path = retrieval_dir / "cli_search_similar_chunks_existing_fn.py"

spec = spec_from_file_location("existing_fn", str(module_path))
existing_fn = module_from_spec(spec)
spec.loader.exec_module(existing_fn)

SearchSimilarChunksClient = existing_fn.SearchSimilarChunksClient
print("Loaded:", SearchSimilarChunksClient)

Loaded: <class 'existing_fn.SearchSimilarChunksClient'>


In [24]:
import re
import time
from collections import defaultdict

Mode = str  # "consult" | "resolve"


def _normalize(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s


def build_query_set(user_query: str, mode: Mode) -> list[str]:
    """
    원칙:
    - 단독 키워드 query 금지(메인 의도 덮음)
    - 메인 문장 유지 + 도메인 토큰을 '붙여서' 확장
    - mode별 토큰은 최소만
    """
    base = _normalize(user_query)
    if not base:
        return []

    # 공통 도메인 토큰(너무 일반적인 '상담' 같은 단독 토큰은 X)
    domain_tokens = [
        "하자",
        "초기불량",
        "고장",
        "AS",
        "수리",
        "교환",
        "환불",
        "반품",
        "배송",
        "지연",
        "누락",
        "파손",
        "오배송",
    ]

    # mode별 최소 토큰
    if mode == "consult":
        mode_tokens = ["절차", "가능", "기간", "증빙"]
    else:
        mode_tokens = ["피해구제", "분쟁조정", "조정 신청", "내용증명"]

    out = []
    seen = set()

    def push(q: str):
        q = _normalize(q)
        if q and q not in seen:
            seen.add(q)
            out.append(q)

    # 1) 메인 쿼리 최우선
    push(base)

    # 2) 메인 + 도메인 토큰 (상위 5개만)
    for t in domain_tokens[:5]:
        push(f"{base} {t}")

    # 3) 메인 + mode 토큰 (2~3개만)
    for t in mode_tokens[:3]:
        push(f"{base} {t}")

    return out[:8]


def retrieve_case_chunks_dense_only(
    user_query: str,
    mode: Mode = "consult",
    top_k: int = 8,
    limit_per_call: int = 5,
):
    """
    search_similar_chunks()만 사용해서 상담/해결/조정 case 청크를 뽑는다.
    return: (results, dbg)
    """
    client = SearchSimilarChunksClient()
    client.connect()
    try:
        cats = ["상담"] if mode == "consult" else ["해결", "조정"]
        queries = build_query_set(user_query, mode)
        if not queries:
            return [], {"mode": mode, "cats": cats, "queries": [], "sql_calls": 0}

        main_q = queries[0]

        sql_calls = 0
        t_total0 = time.time()
        candidates = []

        for cat in cats:
            for q in queries:
                sql_calls += 1
                rows = client.search(
                    query=q,
                    filter_dataset="case",
                    filter_category=cat,
                    result_limit=limit_per_call,
                )
                for r in rows:
                    sim = float(r.similarity)

                    # ✅ 메인 쿼리 우대(미세)
                    score = sim + (0.03 if q == main_q else 0.0)

                    candidates.append(
                        {
                            "chunk_id": r.chunk_id,
                            "dataset_type": r.dataset_type,
                            "category": r.category,
                            "similarity": sim,
                            "score": score,  # ranking용
                            "text": r.text,
                            "_cat": cat,
                            "_q": q,
                            "_is_main_q": (q == main_q),
                        }
                    )
    finally:
        client.close()

    ms_total = (time.time() - t_total0) * 1000

    # 1) chunk_id 기준 dedup: score 최고만
    best = {}
    for r in candidates:
        cid = r["chunk_id"]
        if cid not in best or r["score"] > best[cid]["score"]:
            best[cid] = r

    merged = sorted(best.values(), key=lambda x: x["score"], reverse=True)

    # 2) 다양성: 카테고리별 cap + 메인쿼리 비중 확보
    out = []
    cat_cnt = defaultdict(int)

    cap = max(2, top_k // max(1, len(cats)))

    for r in merged:
        if cat_cnt[r["_cat"]] >= cap:
            continue
        out.append(r)
        cat_cnt[r["_cat"]] += 1
        if len(out) >= top_k:
            break

    dbg = {
        "mode": mode,
        "cats": cats,
        "queries": queries,
        "main_q": main_q,
        "sql_calls": sql_calls,
        "ms_total": round(ms_total, 1),
        "n_candidates": len(candidates),
        "n_dedup": len(merged),
        "cat_cnt": dict(cat_cnt),
    }
    return out, dbg

In [28]:
query = "전화권유판매 철회"
results, dbg = retrieve_case_chunks_dense_only(query, mode="consult", top_k=8)

print("dbg:", dbg)
for i, r in enumerate(results, 1):
    print(
        f"[{i}] cat={r['category']} sim={r['similarity']:.4f} id={r['chunk_id']} (q={r['_q']})"
    )
    print(r["text"][:600])
    print("-" * 60)

dbg: {'mode': 'consult', 'cats': ['상담'], 'queries': ['전화권유판매 철회', '전화권유판매 철회 하자', '전화권유판매 철회 초기불량', '전화권유판매 철회 고장', '전화권유판매 철회 AS', '전화권유판매 철회 수리', '전화권유판매 철회 절차', '전화권유판매 철회 가능'], 'main_q': '전화권유판매 철회', 'sql_calls': 8, 'ms_total': 5449.1, 'n_candidates': 40, 'n_dedup': 12, 'cat_cnt': {'상담': 8}}
[1] cat=상담 sim=0.6195 id=crawl_semantic_상담_8574_full_1 (q=전화권유판매 철회 가능)
질문: - 2일 전 특판 대리점에서 6만원 보조받아 30만원에 휴대폰을 구매할 수 있다는 설명을 듣고 동의함.- 알뜰할인요금제를 사용할 경우 일정 금액 이상 사용 시 1만 원 이상 할인받을 수 있다고 했는데 잘못 거래한 것 같아 철회 차 연락하니 거부함.- 아직 단말기는 배송되지 않은 상황인데 철회가 가능한지?

답변: - 전화권유 판매로 판매사원이 거짓으로 구두상 전화권유로 체결한 계약은 방문판매 등에 관한 법률에 의거 계약서를 교부받은 날부터 14일, 계약서를 교부받은 때보다 재화 등의 공급이 늦게 이루어진 경우에는 재화 등을 공급받거나 공급이 개시된 날부터 14일 이내 당해 계약에 관한 청약철회를 할 수 있음.- 단말기가 배송되어 오면 사업자에게 서면으로 내용증명 발송해야 함.
------------------------------------------------------------
[2] cat=상담 sim=0.6192 id=crawl_semantic_상담_3621_full_1 (q=전화권유판매 철회 절차)
질문: - 이동전화를 사용하고 있는데 텔레마케터로부터 경품에 당첨되어 단말기를 무료 제공받을 수 있다는 설명을 듣고 동의함.- 2차 확인전화가 걸려 왔을 때 처음 설명내용과 동일할 것이라 생각하고 동